In [1]:
# Cài thư viện cho kernel hiện tại (Kaggle/Colab/local đều chạy được)
%pip install -q underthesea

Note: you may need to restart the kernel to use updated packages.


# Bài 1 – Tiền xử lý văn bản & TF-IDF

**Mục tiêu:** hiểu máy tính biến chữ thành số như thế nào ở cách đơn giản nhất.

**Khái niệm:**
- **Token**: đơn vị nhỏ nhất ta xử lý (từ, hoặc mảnh từ). Tiếng Việt khó hơn tiếng Anh vì từ có thể gồm nhiều tiếng ("sổ đỏ", "giấy chứng nhận").
- **Chuẩn hoá**: đưa chữ về dạng thống nhất (Unicode NFC, chữ thường, bỏ ký tự thừa).
- **Bag-of-Words**: đếm mỗi từ xuất hiện bao nhiêu lần, bỏ qua thứ tự.
- **TF-IDF**: như Bag-of-Words nhưng từ xuất hiện ở *mọi* văn bản ("của", "là") được cho điểm thấp, từ đặc trưng cho ít văn bản điểm cao.

In [2]:
import unicodedata, re

def chuan_hoa(s):
    s = unicodedata.normalize('NFC', s)   # gộp dấu tiếng Việt về một dạng
    s = s.lower()
    s = re.sub(r'[^\w\s]', ' ', s)         # bỏ dấu câu
    s = re.sub(r'\s+', ' ', s).strip()
    return s

chuan_hoa('  Giấy CHỨNG NHẬN quyền sử dụng đất!!! ')

'giấy chứng nhận quyền sử dụng đất'

## Vì sao cần NFC?
Chữ "ế" có thể được lưu bằng 1 ký tự hoặc 2 ký tự ghép. Nhìn giống hệt nhau nhưng máy coi là khác nhau.

In [3]:
a = 'ế'
b = unicodedata.normalize('NFD', a)
print(a == b, len(a), len(b))

False 1 3


## Tách từ tiếng Việt

In [4]:
from underthesea import word_tokenize
word_tokenize('Tôi học xử lý ngôn ngữ tự nhiên tại Hà Nội')

['Tôi', 'học', 'xử lý', 'ngôn ngữ', 'tự nhiên', 'tại', 'Hà Nội']

**Câu hỏi 1:** Nếu không tách từ mà chỉ tách theo dấu cách thì "ngôn ngữ" thành mấy token? Điều đó có thể gây vấn đề gì?

*(Viết câu trả lời vào ô này.)*

## Bag-of-Words và TF-IDF với scikit-learn

In [5]:
docs = [
    'giấy chứng nhận quyền sử dụng đất',
    'hợp đồng chuyển nhượng quyền sử dụng đất',
    'đơn đăng ký biến động đất đai',
    'giấy chứng minh nhân dân',
]

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import pandas as pd

cv = CountVectorizer()
bow = cv.fit_transform(docs)
print(pd.DataFrame(bow.toarray(), columns=cv.get_feature_names_out()))

   biến  chuyển  chứng  dân  dụng  giấy  hợp  ký  minh  nhân  nhượng  nhận  \
0     0       0      1    0     1     1    0   0     0     0       0     1   
1     0       1      0    0     1     0    1   0     0     0       1     0   
2     1       0      0    0     0     0    0   1     0     0       0     0   
3     0       0      1    1     0     1    0   0     1     1       0     0   

   quyền  sử  đai  đăng  đơn  đất  đồng  động  
0      1   1    0     0    0    1     0     0  
1      1   1    0     0    0    1     1     0  
2      0   0    1     1    1    1     0     1  
3      0   0    0     0    0    0     0     0  


In [6]:
tf = TfidfVectorizer()
m = tf.fit_transform(docs)
pd.DataFrame(m.toarray(), columns=tf.get_feature_names_out()).round(2)

,biến,chuyển,chứng,dân,dụng,giấy,hợp,ký,minh,nhân,nhượng,nhận,quyền,sử,đai,đăng,đơn,đất,đồng,động
0,0.0,0.0,0.37,0.00,0.37,0.37,0.0,0.0,0.00,0.00,0.0,0.47,0.37,0.37,0.0,0.0,0.0,0.30,0.0,0.0
1,0.0,0.4,0.00,0.00,0.31,0.00,0.4,0.0,0.00,0.00,0.4,0.00,0.31,0.31,0.0,0.0,0.0,0.25,0.4,0.0
2,0.4,0.0,0.00,0.00,0.00,0.00,0.0,0.4,0.00,0.00,0.0,0.00,0.00,0.00,0.4,0.4,0.4,0.25,0.0,0.4
3,0.0,0.0,0.38,0.49,0.00,0.38,0.0,0.0,0.49,0.49,0.0,0.00,0.00,0.00,0.0,0.0,0.0,0.00,0.0,0.0


**Quan sát:** từ "quyền", "sử", "dụng" xuất hiện ở 2/4 văn bản nên điểm thấp hơn "biến", "động" chỉ xuất hiện ở 1 văn bản.

## Đo độ giống nhau giữa 2 văn bản
**Cosine similarity**: 1 = giống hệt, 0 = không có từ chung.

In [7]:
from sklearn.metrics.pairwise import cosine_similarity
print(pd.DataFrame(cosine_similarity(m)).round(2))

      0     1     2     3
0  1.00  0.43  0.08  0.28
1  0.43  1.00  0.06  0.00
2  0.08  0.06  1.00  0.00
3  0.28  0.00  0.00  1.00


## Bài tập
1. Thêm 2 văn bản mới của riêng bạn (ví dụ từ công việc) vào `docs`, xem ma trận similarity thay đổi thế nào.
2. Văn bản nào giống "giấy chứng nhận quyền sử dụng đất" nhất? Có hợp lý không?
3. Thử bỏ bước tách từ, so sánh kết quả với `TfidfVectorizer(ngram_range=(1,2))`. Bigram giúp gì cho tiếng Việt?
4. **Thử thách:** viết hàm `tim_giong_nhat(cau_hoi, docs)` trả về văn bản giống câu hỏi nhất. Đây là dạng đơn giản nhất của search engine.